In [ ]:
# import the required libraries
import pandas as pd
import torch
import json

In [ ]:
# read the input data (embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the embedded keywords-title dataset is the dataset 
formed by merging the embedded keywords and embedded title dataset
"""
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/embedded-data/mkg_title_dataset_gpt.csv")

In [ ]:
movies_keywords_genres.head(1)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 7.77254701e-02 -5.38552813e-02 -5.79311967e-...,[-6.28197715e-02 -2.74479061e-01 -3.25453132e-...


In [ ]:
movies_keywords_genres.shape[0]

234753

In [ ]:
# Extract unique genres from the entire dataset
all_genres = [genre.strip() for genres in movies_keywords_genres['genres'] if pd.notna(genres) for genre in genres.split(',')]
unique_genres = set(all_genres)

# Create a dictionary to map genres to numeric indices
genre_index_mapping = {genre: index + 1 for index, genre in enumerate(unique_genres)}

# Print the mapping of indices to genres
print("Genre Index Mapping: \n", genre_index_mapping)

json_string = json.dumps(genre_index_mapping, indent=2)
file_path = '/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/genre_mapping.txt'
with open(file_path, 'w') as file:
    file.write(json_string)


Genre Index Mapping: 
 {'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}


In [ ]:
# Map genres to numeric indices in the DataFrame and create a new column 'genre_indices'
movies_keywords_genres['genres_indices'] = movies_keywords_genres['genres'].apply(lambda genres: [genre_index_mapping[genre.strip()] for genre in genres.split(',') if pd.notna(genres)])

In [ ]:
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding,genres_indices
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 7.77254701e-02 -5.38552813e-02 -5.79311967e-...,[-6.28197715e-02 -2.74479061e-01 -3.25453132e-...,"[11, 8]"
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[ 9.63690951e-02 6.21698722e-02 -5.16188264e-...,[ 1.28780708e-01 1.27590522e-01 -6.06716909e-...,[8]
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,[-1.05129234e-01 1.31294459e-01 -1.72948554e-...,[-4.49490517e-01 -5.11738248e-02 7.86833838e-...,"[8, 16]"
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,[-2.57868499e-01 6.26966059e-02 -7.99108744e-...,[ 2.59599864e-01 1.75047010e-01 -2.48356387e-...,"[19, 8, 10]"
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,[ 2.70989593e-02 1.46407694e-01 -3.80637795e-...,[ 8.48557949e-02 1.20956279e-01 1.18412793e-...,"[8, 7, 4]"


In [ ]:
# convert the embedded column array to embedded string and clean the keywords_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_keywords_genres['keywords_embedding_string'] = movies_keywords_genres['keywords_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

<ipython-input-8-481dfa4f3b59>:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies_keywords_genres['keywords_embedding_string'] = movies_keywords_genres['keywords_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')


In [ ]:
movies_keywords_genres.loc[0, 'keywords_embedding_string']

' 7.77254701e-02 -5.38552813e-02 -5.79311967e-01  4.74915989e-02 -2.42409214e-01 -1.06283017e-01 -1.02920139e+00  1.92754731e-01  1.32103279e-01  1.22597031e-01  4.21293765e-01  9.23107266e-02 -1.36534885e-01  1.46162793e-01  9.97747630e-02  1.70870438e-01  1.67087968e-02 -2.28376072e-02  7.90126920e-01  2.47801781e-01  3.48790467e-01 -2.53627151e-01  2.13639233e-02  7.41439387e-02  1.38777196e-01  2.63491958e-01  1.37033910e-01  1.28788665e-01 -2.20236108e-01 -1.68634519e-01  1.66344792e-02 -3.49217579e-02  9.52402130e-02 -5.68287075e-02 -1.20635219e-01  1.66217044e-01  5.75536728e+01 -1.30456224e-01  1.34181082e-01 -5.75820263e-03  1.52426600e-01  5.02094403e-02 -3.88921471e-03 -1.92784399e-01  4.05713432e-02 -4.28355664e-01 -9.22270715e-02 -7.62661397e-02 -1.04415920e-02 -2.38590389e-01  3.70137006e-01  1.04515098e-01  9.61054042e-02  1.89142954e-02  1.96001917e-01  4.49541397e-02  2.82831609e-01  4.32456248e-02 -2.52948225e-01  2.18558777e-02 -2.09164679e-01  7.13285431e-02  6.4676

In [ ]:
# convert the embedded array to embedded string and clean the original_title_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_keywords_genres['original_title_embedding_string'] = movies_keywords_genres['original_title_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

<ipython-input-10-c287eab7b5aa>:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies_keywords_genres['original_title_embedding_string'] = movies_keywords_genres['original_title_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')


In [ ]:
movies_keywords_genres.loc[0, 'original_title_embedding_string']

'-6.28197715e-02 -2.74479061e-01 -3.25453132e-01  4.16513324e-01  3.79798785e-02 -1.07708775e-01  3.06926918e+00 -2.46200636e-01 -3.52739722e-01  3.20552558e-01 -2.86789495e-03 -1.85362399e-01 -9.57644954e-02 -2.27944061e-01  5.76466322e-01 -2.48026550e-01  1.42806947e-01 -4.23650891e-01  4.32069987e-01  8.03884566e-01  8.62231031e-02  2.60390610e-01  3.95544976e-01  7.78755918e-02  1.73946425e-01 -8.81615281e-02 -4.09404665e-01  1.58329144e-01  3.35642338e-01  2.49475554e-01 -2.64690798e-02  1.17590092e-01  1.50934622e-01 -8.28338191e-02  2.54233390e-01  2.93531746e-01  6.63708572e+01  5.45789041e-02  3.67209435e-01  3.36284190e-01 -1.89085081e-01  1.44132838e-01  7.08689913e-02 -2.25469410e-01 -1.06010079e-01  4.87182885e-01 -1.62828028e-01 -3.67530972e-01  2.29100034e-01  1.17431439e-01 -3.26384306e-01 -1.58609059e-02  3.99926186e-01  1.97719336e-01  5.13950765e-01  6.60153151e-01 -9.04711057e-03 -3.10007989e-01  1.58675481e-02 -2.38043889e-01 -2.96100616e-01  2.48088777e-01  1.6379

In [ ]:
movies_keywords_genres.head(2)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 7.77254701e-02 -5.38552813e-02 -5.79311967e-...,[-6.28197715e-02 -2.74479061e-01 -3.25453132e-...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.30,26.0,1115.0,friendship berlin germany unsociability loss o...,[ 9.63690951e-02 6.21698722e-02 -5.16188264e-...,[ 1.28780708e-01 1.27590522e-01 -6.06716909e-...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...


In [ ]:
# drop the embedded array columns for keywords and original_title
drop_columns = ['keywords_embedding', 'original_title_embedding']
movies_keywords_genres = movies_keywords_genres.drop(columns=drop_columns)
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...


In [ ]:
# remove any NaN rows & columns
nan_count = movies_keywords_genres['keywords_embedding_string'].isnull().sum(); print(f"The column 'keywords_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['original_title_embedding_string'].isnull().sum(); print(f"The column 'original_title_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
movies_keywords_genres.dropna(subset=['keywords_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['original_title_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['genres'], inplace=True)

The column 'keywords_embedding_string' has 34 NaN values.
The column 'original_title_embedding_string' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
# export the cleaned embedded keywords and title dataset to a .csv file
movies_keywords_genres.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/embedded-data/final_mkg_dataset_gpt.csv', index=False)